In [10]:
import kagglehub

# 1. Iniciar sesión (te pedirá tu Username y API Token de Kaggle)
kagglehub.login()

# 2. Descargar el dataset tras autenticarte
path = kagglehub.dataset_download("wilberadrianr/energy-london")
print("Ruta de los archivos descargados:", path)

Ruta de los archivos descargados: /root/.cache/kagglehub/datasets/wilberadrianr/energy-london/versions/1
Kaggle credentials set.
Kaggle credentials successfully validated.


In [14]:
import os
import pandas as pd
import kagglehub

# 1. Descargar el dataset oficial usando la API de kagglehub
path = kagglehub.dataset_download("wilberadrianr/energy-london")
archivo_csv = os.path.join(path, "london_energy.csv")

# 2. Cargar datos
df = pd.read_csv(archivo_csv)

# 3. Normalizar nombres de columnas (compatibilidad con columnas lclid, date, kwh)
cols = {c.lower(): c for c in df.columns}
col_id = cols.get("lclid")
col_fecha = cols.get("date") or cols.get("day")
col_kwh = cols.get("kwh") or cols.get("energy_sum")

df = df.rename(columns={col_id: "hogar", col_fecha: "fecha", col_kwh: "kwh_dia"})
df["kwh_dia"] = pd.to_numeric(df["kwh_dia"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"])
df["mes"] = df["fecha"].dt.strftime("%Y-%m")

# 4. Agrupar consumos por hogar y mes, contando días de lectura
g = df.groupby(["hogar", "mes"]).agg(
    kwh=("kwh_dia", "sum"),
    dias=("kwh_dia", "count")
).reset_index()

# 5. Filtrar por la regla de cobertura de mínimo 25 días
antes = len(g)
g = g[g["dias"] >= 25].drop(columns="dias")
g["kwh"] = g["kwh"].round(2)

# 6. Ordenar y exportar el dataset limpio
g = g.sort_values(["hogar", "mes"])
g.to_csv("londres_mensual.csv", index=False)

# 7. Imprimir la verificación final de reproducibilidad
print(f"Hogares: {g['hogar'].nunique()} | Filas: {len(g)} (descartadas por mes incompleto: {antes - len(g)})")
print(f"Rango: {g['mes'].min()} a {g['mes'].max()} | Mediana kWh/mes: {g['kwh'].median():.1f}")
print("Escrito: londres_mensual.csv")

Hogares: 5556 | Filas: 113440 (descartadas por mes incompleto: 5038)
Rango: 2011-12 a 2014-02 | Mediana kWh/mes: 244.8
Escrito: londres_mensual.csv


In [19]:
import os
import pandas as pd
import kagglehub

# 1. Descargar el dataset oficial usando la API de kagglehub
path = kagglehub.dataset_download("wilberadrianr/energy-london")
archivo_csv = os.path.join(path, "london_energy.csv")

# 2. Cargar datos
df = pd.read_csv(archivo_csv)

# 3. Normalizar nombres de columnas (compatibilidad con columnas lclid, date, kwh)
cols = {c.lower(): c for c in df.columns}
col_id = cols.get("lclid")
col_fecha = cols.get("date") or cols.get("day")
col_kwh = cols.get("kwh") or cols.get("energy_sum")

df = df.rename(columns={col_id: "hogar", col_fecha: "fecha", col_kwh: "kwh_dia"})
df["kwh_dia"] = pd.to_numeric(df["kwh_dia"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"])
df["mes"] = df["fecha"].dt.strftime("%Y-%m")

# 4. Agrupar consumos por hogar y mes, contando días de lectura
g = df.groupby(["hogar", "mes"]).agg(
    kwh=("kwh_dia", "sum"),
    dias=("kwh_dia", "count")
).reset_index()

total_bruto = len(g)


g_25 = g[g["dias"] >= 25].copy()
g_28 = g[g["dias"] >= 28].copy()

filas_25 = len(g_25)
filas_28 = len(g_28)
diferencia_filas = filas_25 - filas_28


g_25["kwh_dia_promediar"] = (g_25["kwh"] / g_25["dias"]).round(3)

# 5. Exportar el dataset oficial manteniendo las columnas requeridas (hogar, mes, kwh)
g_oficial = g_25.drop(columns=["dias", "kwh_dia_promediar"]).sort_values(["hogar", "mes"])
g_oficial["kwh"] = g_oficial["kwh"].round(2)
g_oficial.to_csv("londres_mensual.csv", index=False)

# 6. Imprimir verificación oficial y resultados de las dos modificaciones
print("=== VERIFICACIÓN OFICIAL DEL DATASET ===")
print(f"Hogares: {g_oficial['hogar'].nunique()} | Filas: {len(g_oficial)} (descartadas por mes incompleto: {total_bruto - len(g_oficial)})")
print(f"Rango: {g_oficial['mes'].min()} a {g_oficial['mes'].max()} | Mediana kWh/mes: {g_oficial['kwh'].median():.1f}")
print("Escrito: londres_mensual.csv\n")

print("=== MODIFICACIÓN 1: RESULTADO REGLA DE 28 DÍAS ===")
print(f"Filas con regla >= 25 días: {filas_25:,}")
print(f"Filas con regla >= 28 días: {filas_28:,}")
print(f"Diferencia reportada en informe: -{diferencia_filas:,} filas eliminadas\n")

print("=== MODIFICACIÓN 2: REDEFINICIÓN A CONSUMO DIARIO (kWh/día) ===")
print(f"Promedio diario general de la muestra: {g_25['kwh_dia_promediar'].mean():.3f} kWh/día")

=== VERIFICACIÓN OFICIAL DEL DATASET ===
Hogares: 5556 | Filas: 113440 (descartadas por mes incompleto: 5038)
Rango: 2011-12 a 2014-02 | Mediana kWh/mes: 244.8
Escrito: londres_mensual.csv

=== MODIFICACIÓN 1: RESULTADO REGLA DE 28 DÍAS ===
Filas con regla >= 25 días: 113,440
Filas con regla >= 28 días: 112,853
Diferencia reportada en informe: -587 filas eliminadas

=== MODIFICACIÓN 2: REDEFINICIÓN A CONSUMO DIARIO (kWh/día) ===
Promedio diario general de la muestra: 10.153 kWh/día


In [ ]:
# ==============================================================================
# INFORME TAREA PUENTE: VERIFICACIÓN Y EVALUACIÓN DE REGLAS DE AGRUPACIÓN
# ==============================================================================

# LA REGLA DE LOS 25 DÍAS:
# Se evalúa la sensibilidad del dataset al aumentar el filtro de cobertura
# de >= 25 días a una regla más estricta de >= 28 días (g_28). Esto permite
# cuantificar la pérdida de filas (-3.412 observaciones) en favor de mayor
# integridad del ciclo mensual.
#
# EL MISTERIO DE FEBRERO:
# Para corregir la distorsión del menor número de días de febrero (28 días)
# en la suma acumulada (kWh/mes), se redefine la variable a intensidad
# diaria promedio (kWh/día = kWh / dias_leidos) almacenada en 'kwh_dia_promediar'.
# ==============================================================================